In [6]:
import sqlite3
from sqlite3 import Error
import pandas as pd

## Подключение к бд и получение схемы просмотра страниц таблицы

In [7]:
def create_connection(path):
    connection = None
    try: 
        connection = sqlite3.connect(path)
    except Error as e:
        print(f"The error {e} occured")
    return connection

connection = create_connection("../checking-logs.sqlite")
df = pd.io.sql.read_sql("PRAGMA table_info(pageviews);", connection)

df


,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,uid,TEXT,0,None,0
2,2,datetime,TIMESTAMP,0,None,0


In [8]:
part_of_table = pd.read_sql("SELECT * FROM pageviews LIMIT 10", connection)

part_of_table

,index,uid,datetime
0,0,admin_1,2020-04-17 12:01:08.463179
1,1,admin_1,2020-04-17 12:01:23.743946
2,2,admin_3,2020-04-17 12:17:39.287778
3,3,admin_3,2020-04-17 12:17:40.001768
4,4,admin_1,2020-04-17 12:27:30.646665
5,5,admin_1,2020-04-17 12:35:44.884757
6,6,admin_1,2020-04-17 12:35:52.735016
7,7,admin_3,2020-04-17 12:36:21.401412
8,8,admin_3,2020-04-17 12:36:22.023355
9,9,admin_1,2020-04-17 13:55:19.129243


## Получение подтаблички
* используются только uid и datetime
* используются только пользовательские данные (user_*), а не данные администратора
* он отсортиван по uid в порядке возрастания
* столбец индекса - дата и время
* datetime преобразуется в DatetimeIndex
* имя фрейма данных - просмотры страниц

In [9]:
pageviews = pd.read_sql("SELECT uid, datetime FROM pageviews WHERE uid LIKE 'user_%' ORDER BY uid;", connection)
pageviews["datetime"] = pd.to_datetime(pageviews["datetime"] )
pageviews.set_index("datetime")


,uid
datetime,
2020-04-26 21:53:59.624136,user_1
2020-04-26 22:06:19.478143,user_1
2020-04-26 22:12:09.614497,user_1
2020-04-30 19:29:01.831635,user_1
2020-05-05 20:26:32.894852,user_1
...,...
2020-04-29 16:51:21.877630,user_30
2020-05-09 20:30:47.034282,user_30
2020-05-22 11:30:18.368990,user_5


In [11]:
pageviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 987 entries, 0 to 986
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   uid       987 non-null    object        
 1   datetime  987 non-null    datetime64[ns]
dtypes: datetime64[ns](1), object(1)
memory usage: 15.5+ KB


## Закрытие соединения с бд

In [10]:
connection.close()